# Pandas Interview Questions — Drills on Real Data

Every drill below runs on the repo's shipped datasets (Titanic + IPL matches) or a small synthetic frame, so every answer you rehearse has been *executed* — the exact muscle memory you want in a live screen where the interviewer is watching your cursor.

**Rehearsal protocol:** read the task, write your one-liner on paper, then compare. In interviews, always run `df.head()`, `df.dtypes`, and `df.shape` *before* touching anything — say that out loud, it's free points.

## Setup — the data we'll use

In [1]:
import numpy as np
import pandas as pd

titanic = pd.read_csv("../../5. Pushing it with Projects/End-to-End ML Project/titanic.csv")
matches = pd.read_csv("../../1. Testing the waters/3. Learn Pandas/matches.csv")

print("titanic:", titanic.shape, "| matches:", matches.shape)
titanic.head(3)

titanic: (891, 12) | matches: (636, 18)


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S


## Drill 1 — The Filter-Combo Question (warm-up)

*Women in 3rd class who survived, showing name and age only, sorted by age descending.*

In [2]:
result = (titanic
    .loc[(titanic.Sex == "female") & (titanic.Pclass == 3) & (titanic.Survived == 1),
         ["Name", "Age"]]
    .sort_values("Age", ascending=False))
print(f"{len(result)} rows")
result.head(5)

72 rows


,Name,Age
483,"Turkula, Mrs. (Hedwig)",63.0
25,"Asplund, Mrs. Carl Oscar (Selma Augusta Emilia...",38.0
559,"de Messemaeker, Mrs. Guillaume Joseph (Emma)",36.0
279,"Abbott, Mrs. Stanton (Rosa Hunt)",35.0
85,"Backstrom, Mrs. Karl Alfred (Maria Mathilda Gu...",33.0


**Points to make:** `.loc` takes *labels/booleans*, `.iloc` takes *positions*; combine filters with `&`/`|` **and parentheses** (operator precedence — `and` raises, unparenthesized `&` silently misparses). Chaining with parentheses keeps it readable and lets you comment out stages.

## Drill 2 — GroupBy: survival rate by class and sex

*The canonical split-apply-combine.* Say the phrase "split, apply, combine" — it's the vocabulary interviewers listen for — and mention `agg` for multiple statistics in one pass.

In [3]:
surv = titanic.groupby(["Pclass", "Sex"], observed=True)["Survived"].agg(["mean", "count"])
surv["mean"] = surv["mean"].round(3)
surv

mean  count
Pclass Sex                 
1      female  0.968     94
       male    0.369    122
2      female  0.921     76
       male    0.157    108
3      female  0.500    144
       male    0.135    347

## Drill 3 — `transform` vs `apply`: fill age with the group mean

*The classic follow-up:* "now fill missing ages with the mean age **of their class and sex**, without a loop."

`transform` returns a series **aligned to the original index** (same length) — that's the one you want. `apply` returns one value *per group*. Wrong-tool use here is the most common pandas screen failure.

In [4]:
n_missing = titanic["Age"].isna().sum()
group_means = titanic.groupby(["Pclass", "Sex"], observed=True)["Age"].transform("mean")

filled = titanic.assign(age_filled=titanic["Age"].fillna(group_means))
print(f"missing ages before: {n_missing}, after: {filled['age_filled'].isna().sum()}")
filled.loc[titanic['Age'].isna(), ['Pclass', 'Sex', 'age_filled']].head(4)

missing ages before: 177, after: 0


,Pclass,Sex,age_filled
5,3,male,26.507589
17,2,male,30.740707
19,3,female,21.750000
26,3,male,26.507589


## Drill 4 — Melt and Pivot: reshape like it's nothing

*Convert wide to long and back.* `melt` unpivots columns into variable/value rows; `pivot_table` re-widens with aggregation built in. Know that `pivot` fails on duplicate index pairs, `pivot_table` aggregates them.

In [5]:
wide = (titanic
    .groupby(["Pclass", "Sex"], observed=True)["Survived"].mean()
    .unstack()                              # index -> columns
    .round(3))
print("wide (pivot):"); display(wide)

long = wide.reset_index().melt(id_vars="Pclass", var_name="Sex", value_name="survival_rate")
print("long (melt):"); display(long)

wide (pivot):


Sex,female,male
Pclass,,
1,0.968,0.369
2,0.921,0.157
3,0.500,0.135


long (melt):


,Pclass,Sex,survival_rate
0,1,female,0.968
1,2,female,0.921
2,3,female,0.500
3,1,male,0.369
4,2,male,0.157
5,3,male,0.135


## Drill 5 — Joins: how many matches did each IPL venue host?

Merge questions are really *key-integrity* questions. State the checks before joining: duplicate keys? nulls? inner vs left semantics? Interviewers watch for `validate=` and `how=` being chosen deliberately.

In [6]:
print("matches rows:", len(matches), "| unique seasons:", matches["season"].nunique())

venue_counts = (matches
    .groupby("venue")["id"]
    .count()
    .sort_values(ascending=False))
venue_counts.head(5)

matches rows: 636 | unique seasons: 10


venue
M Chinnaswamy Stadium                        66
Eden Gardens                                 61
Feroz Shah Kotla                             60
Wankhede Stadium                             57
Rajiv Gandhi International Stadium, Uppal    49
Name: id, dtype: int64

In [7]:
# a real merge drill: season-level frame joined back onto matches
season_stats = matches.groupby("season").agg(
    n_matches=("id", "count"),
    avg_first_innings=("first_innings_score" if "first_innings_score" in matches.columns else "id", "mean" if "first_innings_score" in matches.columns else "count"),
)
print("columns available:", [c for c in matches.columns][:12])
# use actual numeric column: win_by_runs
season_stats = matches.groupby("season").agg(
    n_matches=("id", "count"),
    avg_win_by_runs=("win_by_runs", "mean"),
).round(1)
enriched = matches.merge(season_stats, on="season", how="left", validate="many_to_one")
print("merge check: rows unchanged ->", len(enriched) == len(matches))
enriched[["season", "venue", "n_matches", "avg_win_by_runs"]].head(3)

columns available: ['id', 'season', 'city', 'date', 'team1', 'team2', 'toss_winner', 'toss_decision', 'result', 'dl_applied', 'winner', 'win_by_runs']
merge check: rows unchanged -> True


,season,venue,n_matches,avg_win_by_runs
0,2017,"Rajiv Gandhi International Stadium, Uppal",59,13.4
1,2017,Maharashtra Cricket Association Stadium,59,13.4
2,2017,Saurashtra Cricket Association Stadium,59,13.4


**The sentence to say:** *"I merged with `how='left'` and `validate='many_to_one'` because the season table has one row per season; validate makes pandas enforce that assumption instead of silently exploding the row count."*

## Drill 6 — Rolling Windows: the moving-average one-liner

*Rolling 3-match average of win margins per team.* The trap: rolling **across the whole frame** mixes teams — sort and group first, then roll.

In [8]:
margins = (matches
    .query("win_by_runs > 0")
    .groupby(["season", "winner"])["win_by_runs"]
    .mean()
    .rename("avg_margin")
    .reset_index())

# rolling 3-season average of each team's winning margin
margins["margin_ma3"] = (margins
    .groupby("winner")["avg_margin"]
    .transform(lambda s: s.rolling(3, min_periods=1).mean())
    .round(1))
margins[margins["winner"] == "Mumbai Indians"].tail(6)

/tmp/ipykernel_1500/2347895573.py:2: UserWarning: numexpr version '2.14.1' can silently return incorrect results and will not be used by pandas. Install numexpr 2.14.2 or newer to re-enable numexpr acceleration.
  .query("win_by_runs > 0")


,season,winner,avg_margin,margin_ma3
35,2012,Mumbai Indians,18.333333,29.8
43,2013,Mumbai Indians,35.333333,26.0
51,2014,Mumbai Indians,19.666667,24.4
59,2015,Mumbai Indians,20.000000,25.0
67,2016,Mumbai Indians,52.500000,30.7
74,2017,Mumbai Indians,42.500000,38.3


## Drill 7 — `value_counts` tricks interviewers expect

Normalized counts, dropping nulls deliberately, binning with `cut`, and top-N with a tie-aware head.

In [9]:
print("survived proportion:")
print(titanic["Survived"].value_counts(normalize=True).round(3).to_string())

print()
print("age bands (binned):")
print(pd.cut(titanic["Age"], bins=[0, 12, 18, 35, 60, 100],
             labels=["child", "teen", "adult", "middle", "senior"])
      .value_counts().to_string())

survived proportion:
Survived
0    0.616
1    0.384

age bands (binned):
Age
adult     358
middle    195
teen       70
child      69
senior     22


## Drill 8 — The Apply-With-Function Question, and When Not To

*Title extraction: pull 'Mr', 'Mrs', 'Miss' from the name column.* Write the function, then say the senior line: **apply is a Python-level loop — if a vectorized equivalent exists, use it.** Then prove it with a timing.

In [10]:
def get_title(name):
    return name.split(",")[1].split(".")[0].strip().lower()

t_col = titanic["Name"].map(get_title)      # map == apply on a Series
print(t_col.value_counts().head(6).to_string())

Name
mr        517
miss      182
mrs       125
master     40
dr          7
rev         6


In [11]:
import timeit
s = titanic["Name"]

t_map = timeit.timeit(lambda: s.map(get_title), number=50)
t_str = timeit.timeit(lambda: s.str.split(",").str[1].str.split(".").str[0].str.strip().str.lower(), number=50)
print(f".map(function) : {t_map:.3f}s   <- Python loop per row")
print(f".str vectorized: {t_str:.3f}s   <- compiled; usually several times faster")

.map(function) : 0.024s   <- Python loop per row
.str vectorized: 0.068s   <- compiled; usually several times faster


## Drill 9 — Missing-Data Report in Four Lines

A "profile the nulls" task appears in nearly every pandas screen. The matrix form (count + percent) is the answer they want to see produced quickly.

In [12]:
nulls = (pd.DataFrame({"n_missing": titanic.isna().sum(),
                      "pct_missing": titanic.isna().mean().round(3) * 100})
         .query("n_missing > 0")
         .sort_values("n_missing", ascending=False))
nulls

,n_missing,pct_missing
Cabin,687,77.1
Age,177,19.9
Embarked,2,0.2


## Drill 10 — Datetime Features, the ML-Adjacent Classic

*Extract features from a timestamp column.* Interviewers check: parse once with `pd.to_datetime`, then use `.dt` accessor — never string slicing.

In [13]:
orders = pd.DataFrame({
    "order_id": range(1, 9),
    "ts": pd.date_range("2026-01-01", periods=8, freq="19h"),
    "amount": [120, np.nan, 340, 90, 250, 400, np.nan, 310],
})
orders["dow"] = orders["ts"].dt.day_name()
orders["hour"] = orders["ts"].dt.hour
orders["is_weekend"] = orders["ts"].dt.dayofweek >= 5

# forward-fill amounts within the same day (leakage-safe usage: past->present)
orders["amount_ffill"] = orders["amount"].ffill()
orders

/tmp/ipykernel_1500/1232124453.py:8: Pandas4Warning: Series.dt.dayofweek is deprecated and will be removed in a future version. Use Series.dt.day_of_week instead.
  orders["is_weekend"] = orders["ts"].dt.dayofweek >= 5


,order_id,ts,amount,dow,hour,is_weekend,amount_ffill
0,1,2026-01-01 00:00:00,120.0,Thursday,0,False,120.0
1,2,2026-01-01 19:00:00,NaN,Thursday,19,False,120.0
2,3,2026-01-02 14:00:00,340.0,Friday,14,False,340.0
3,4,2026-01-03 09:00:00,90.0,Saturday,9,True,90.0
4,5,2026-01-04 04:00:00,250.0,Sunday,4,True,250.0
5,6,2026-01-04 23:00:00,400.0,Sunday,23,True,400.0
6,7,2026-01-05 18:00:00,NaN,Monday,18,False,400.0
7,8,2026-01-06 13:00:00,310.0,Tuesday,13,False,310.0


## Drill 11 — The "Top-N per Group" Question

*Best 2 matches by margin for each team.* Two idioms: `sort + groupby.head` (simple) and `nlargest` per group. Sort-then-head is easier to write bug-free on a whiteboard.

In [14]:
top2 = (matches
    .query("win_by_runs > 0")
    .sort_values("win_by_runs", ascending=False)
    .groupby("winner", as_index=False)
    .head(2)
    [["winner", "season", "win_by_runs"]])
top2.head(8)

,winner,season,win_by_runs
43,Mumbai Indians,2017,146
619,Royal Challengers Bangalore,2016,144
59,Kolkata Knight Riders,2008,140
556,Royal Challengers Bangalore,2015,138
295,Kings XI Punjab,2011,111
114,Rajasthan Royals,2008,105
182,Mumbai Indians,2010,98
541,Chennai Super Kings,2015,97


## Drill 12 — `query`, `assign`, and Method-Chaining Style

Modern pandas screens increasingly expect fluent chaining — no intermediate variables, no `inplace=True`. `query` reads like SQL; `assign` adds columns inside a chain.

In [15]:
report = (titanic
    .query("Age >= 18 and Survived == 1")
    .assign(age_band=lambda d: pd.cut(d["Age"], [18, 30, 45, 100], labels=["18-30", "30-45", "45+"]))
    .groupby(["Pclass", "age_band"], observed=True)
    .size()
    .rename("n_survivors")
    .reset_index()
    .sort_values(["Pclass", "age_band"]))
report

,Pclass,age_band,n_survivors
0,1,18-30,32
1,1,30-45,46
2,1,45+,30
3,2,18-30,28
4,2,30-45,25
5,2,45+,7
6,3,18-30,36
7,3,30-45,15
8,3,45+,1


## Rapid-Fire: The One-Liners You Must Not Fumble

| Task | The line |
| --- | --- |
| First rows / dtypes / shape | `df.head()`, `df.dtypes`, `df.shape` — say you run these first, always |
| Unique values + count | `df["col"].nunique()`, `df["col"].value_counts()` |
| Percent of nulls per column | `df.isna().mean()` |
| Duplicated rows | `df.duplicated().sum()`; drop with `df.drop_duplicates(subset=[...])` |
| Column rename / select | `df.rename(columns={...})`; `df[["a", "b"]]` |
| Change dtype | `df["col"] = df["col"].astype("int32")` — mention downcasting for memory |
| Conditional column | `np.where(cond, a, b)` for binary, `np.select` for multi-way |
| Cumulative sum | `df.groupby("g")["v"].cumsum()` |
| Rank within group | `df.groupby("g")["v"].rank(method="dense", ascending=False)` |
| Percentile binning | `pd.qcut(df["v"], 4, labels=[...])` |
| Read only needed columns | `pd.read_csv(path, usecols=[...])` — big-file memory answer |
| Chunked reading | `pd.read_csv(path, chunksize=100_000)` then accumulate — the "doesn't fit in RAM" answer |